F6. Семантическая энтропия: разнообразие по содержанию, а не по меткам.....


In [1]:
import sys, os, json, re
from itertools import combinations

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/panel_mechanisms')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
from sklearn.feature_extraction.text import TfidfVectorizer
import statsmodels.api as sm
A = load_arms()
PANEL_REL = {'P-base': PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv', 'P-weak': PANELS_OTHER_DIR['P-weak'] + '/llm_as_judge/' + PANEL_FILE['P-weak'], 'P-mixed': PANELS_OTHER_DIR['P-mixed'] + '/llm_as_judge/' + PANEL_FILE['P-mixed'], 'P-strong': PANELS_OTHER_DIR['P-strong'] + '/llm_as_judge/' + PANEL_FILE['P-strong'], 'homog gpt': HOMOG_DIRS['gpt'] + '/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', 'homog qwen': HOMOG_DIRS['qwen'] + '/llm_as_judge/delphi_homog_qwen_noroles_judge.csv', 'homog gemini': HOMOG_DIRS['gemini'] + '/llm_as_judge/delphi_homog_gemini_noroles_judge.csv', 'homog deepseekv4': HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'}


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


In [2]:
_FENCE = re.compile('^\\s*```(?:json)?\\s*|\\s*```\\s*$', re.I)
_QA = re.compile('"question"\\s*:\\s*"(.*?)"\\s*,\\s*"my_answer"\\s*:\\s*"(.*?)"\\s*,\\s*"rationale"\\s*:\\s*"(.*?)"', re.S)

def _load_json_loose(txt):
    s = _FENCE.sub('', str(txt).strip())
    try:
        return (json.loads(s), 'прямой')
    except Exception:
        pass
    i, j = (s.find('{'), s.rfind('}'))
    if i >= 0 and j > i:
        try:
            return (json.loads(s[i:j + 1]), 'по скобкам')
        except Exception:
            pass
    m = _QA.findall(s)
    if m:
        return ({'sub_questions': [{'question': a, 'my_answer': b, 'rationale': c} for a, b, c in m]}, 'регуляркой')
    return (None, 'не разобрано')

def parse_r1(txt):
    o, _ = _load_json_loose(txt)
    if o is None:
        return None
    qs = o.get('sub_questions') or []
    return {'questions': [str(q.get('question', '')) for q in qs], 'answers': [str(q.get('my_answer', '')) for q in qs], 'rationales': [str(q.get('rationale', '')) for q in qs], 'answer_text': ' '.join((str(q.get('my_answer', '')) for q in qs)), 'full_text': ' '.join((str(q.get('my_answer', '')) + ' ' + str(q.get('rationale', '')) for q in qs)), 'n_q': len(qs)}

def parse_rn(txt):
    o, _ = _load_json_loose(txt)
    if o is None:
        return None
    out = {}
    for a in o.get('answers') or []:
        qid = str(a.get('question_id', ''))
        if qid:
            out[qid] = str(a.get('answer', '')) + ' ' + str(a.get('rationale', ''))
    return out
ROWS, PARSE = ([], [])
for arm, rel in PANEL_REL.items():
    d = rd(rel)
    if d is None:
        continue
    ps = participants(d)
    for p in ps:
        c = 'r1_raw__' + p
        if c not in d.columns:
            continue
        for v, txt in d[c].dropna().items():
            o, how = _load_json_loose(txt)
            PARSE.append({'arm': arm, 'способ': how})
            r = parse_r1(txt)
            if r and r['answer_text'].strip():
                ROWS.append({'arm': arm, 'case': v, 'pid': p, 'n_q': r['n_q'], 'answer_text': r['answer_text'], 'full_text': r['full_text'], 'questions': r['questions']})
R1 = pd.DataFrame(ROWS)
PA = pd.DataFrame(PARSE)
show(PA.pivot_table(index='arm', columns='способ', values='способ', aggfunc='size', fill_value=0), 'Как разобрался JSON ответов участников')
print()
print('ВАЖНО. Часть моделей оборачивает JSON в markdown-ограждения ```json,')
print('часть обрывает вывод. Наивный json.loads терял до 70%% записей у')
print('отдельных плеч (gemini: 208 из 687), что дало бы СМЕЩЁННУЮ выборку —')
print('уцелели бы только аккуратно оформленные, то есть, возможно, более')
print('шаблонные ответы. Разбор с запасными вариантами это устраняет.')
print()
print('разобрано записей R1 (плечо × кейс × участник):', len(R1))
print('плеч: %d, кейсов: %d' % (R1['arm'].nunique(), R1['case'].nunique()))
print('медиана слов в ответах одного участника: %d' % int(R1['answer_text'].map(words).median()))
print('подвопросов на участника: %s' % R1['n_q'].value_counts().sort_index().to_dict())



Как разобрался JSON ответов участников
------------------------------------------------------------
способ            не разобрано  прямой  регуляркой
arm                                               
P-base                       0     685           2
P-mixed                      0     686           1
P-strong                     0     680           7
P-weak                       1     678           8
homog deepseekv4             0     684           0
homog gemini                 0     687           0
homog gpt                    0     629           0
homog qwen                   0     686           1

ВАЖНО. Часть моделей оборачивает JSON в markdown-ограждения ```json,
часть обрывает вывод. Наивный json.loads терял до 70%% записей у
отдельных плеч (gemini: 208 из 687), что дало бы СМЕЩЁННУЮ выборку —
уцелели бы только аккуратно оформленные, то есть, возможно, более
шаблонные ответы. Разбор с запасными вариантами это устраняет.

разобрано записей R1 (плечо × кейс × участник): 5434
пл

медиана слов в ответах одного участника: 120
подвопросов на участника: {1: 1, 3: 1, 6: 2, 7: 9, 8: 5421}


In [3]:
rows = []
for arm, rel in PANEL_REL.items():
    d = rd(rel)
    if d is None:
        continue
    ps = participants(d)
    em = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in ps})
    sc = pd.DataFrame({p: d['r1_subclinical__' + p] for p in ps if 'r1_subclinical__' + p in d.columns})
    full_em = em.dropna()
    full_sc = sc.dropna()
    rows.append({'плечо': arm, 'кейсов': len(d), 'единогласие по срочности (3 знач.)': float((full_em.nunique(axis=1) == 1).mean()) if len(full_em) else np.nan, 'единогласие по специальности (19 знач.)': float((full_sc.nunique(axis=1) == 1).mean()) if len(full_sc) else np.nan})
L = pd.DataFrame(rows).set_index('плечо')
show(L, 'Единогласие на двух уровнях огрубления метки')
print()
print('Уже здесь видно главное: чем тоньше метка, тем меньше согласия.')
print('Разница между колонками — это разнообразие, которое B9 не видел,')
print('потому что считал только по трёхуровневой метке срочности.')



Единогласие на двух уровнях огрубления метки
------------------------------------------------------------
                  кейсов  единогласие по срочности (3 знач.)  единогласие по специальности (19 знач.)
плечо                                                                                                
P-base               229                              0.7181                                   0.7269
P-weak               229                              0.7955                                   0.6455
P-mixed              229                              0.8684                                   0.7237
P-strong             229                              0.7568                                   0.6937
homog gpt            229                              0.8901                                   0.8272
homog qwen           229                              0.9474                                   0.9075
homog gemini         229                              0.9563                 

In [4]:
vec = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=3, max_df=0.7, sublinear_tf=True, stop_words='english')
X = vec.fit_transform(R1['full_text'])
print('матрица TF-IDF:', X.shape)
Xn = X.multiply(1.0 / np.sqrt(X.multiply(X).sum(axis=1)))
Xn = Xn.tocsr()

def case_sims(idx):
    sub = Xn[idx]
    S = (sub @ sub.T).toarray()
    np.fill_diagonal(S, 1.0)
    return S
groups = R1.groupby(['arm', 'case'], observed=True).indices
print('групп (плечо × кейс):', len(groups))
sims = {}
for key, idx in groups.items():
    if len(idx) >= 2:
        sims[key] = case_sims(idx)
allpairs = np.concatenate([S[np.triu_indices_from(S, 1)] for S in sims.values()])
print('попарных сравнений:', len(allpairs))
print('косинусное сходство между участниками: медиана %.3f, Q1 %.3f, Q3 %.3f' % (np.median(allpairs), np.quantile(allpairs, 0.25), np.quantile(allpairs, 0.75)))


матрица TF-IDF: (5434, 30000)
групп (плечо × кейс): 1826


попарных сравнений: 5398
косинусное сходство между участниками: медиана 0.505, Q1 0.427, Q3 0.582


In [5]:
rng = np.random.default_rng(0)
ridx = rng.choice(Xn.shape[0], 4000, replace=False)
sub = Xn[ridx]
between = (sub @ sub.T).toarray()
between = between[np.triu_indices_from(between, 1)]
print('сходство МЕЖДУ случайными парами (разные кейсы): медиана %.3f, 95%% квантиль %.3f' % (np.median(between), np.quantile(between, 0.95)))
print('сходство ВНУТРИ кейса (разные участники):        медиана %.3f, Q1 %.3f, Q3 %.3f' % (np.median(allpairs), np.quantile(allpairs, 0.25), np.quantile(allpairs, 0.75)))
print()
print('РАЗРЫВ ОГРОМЕН: внутри кейса сходство примерно в 15-20 раз выше фонового.')
print('Значит лексика ответа определяется прежде всего ТЕМ, О КАКОМ СЛУЧАЕ РЕЧЬ,')
print('а не тем, согласны ли участники. Двое, обсуждающих одну виньетку, делят')
print('почти весь словарь независимо от того, спорят они или нет.')
print()
print('Следствие: порог, откалиброванный по фону, отправит ВСЕ тройки в один')
print('кластер, энтропия окажется тождественно нулевой, и метрика ничего не')
print('измерит. Проверим это явно, чтобы не быть голословными:')
TAU_BG = float(np.quantile(between, 0.95))
print('   порог по фону составил бы τ = %.3f при внутрикейсовой медиане %.3f' % (TAU_BG, np.median(allpairs)))
print('   доля пар выше такого порога: %.3f — то есть практически все.' % float((allpairs >= TAU_BG).mean()))
print()
print('ПРАВИЛЬНАЯ КАЛИБРОВКА — по распределению сходств ВНУТРИ кейсов.')
print('Порог отвечает на вопрос «похожи ли эти трое сильнее, чем типичная')
print('тройка участников на типичном кейсе», а не «говорят ли они о медицине».')
TAU = float(np.median(allpairs))
print('опорный порог τ = %.3f (медиана внутрикейсовых сходств)' % TAU)


сходство МЕЖДУ случайными парами (разные кейсы): медиана 0.025, 95% квантиль 0.071
сходство ВНУТРИ кейса (разные участники):        медиана 0.505, Q1 0.427, Q3 0.582

РАЗРЫВ ОГРОМЕН: внутри кейса сходство примерно в 15-20 раз выше фонового.
Значит лексика ответа определяется прежде всего ТЕМ, О КАКОМ СЛУЧАЕ РЕЧЬ,
а не тем, согласны ли участники. Двое, обсуждающих одну виньетку, делят
почти весь словарь независимо от того, спорят они или нет.

Следствие: порог, откалиброванный по фону, отправит ВСЕ тройки в один
кластер, энтропия окажется тождественно нулевой, и метрика ничего не
измерит. Проверим это явно, чтобы не быть голословными:
   порог по фону составил бы τ = 0.071 при внутрикейсовой медиане 0.505
   доля пар выше такого порога: 1.000 — то есть практически все.

ПРАВИЛЬНАЯ КАЛИБРОВКА — по распределению сходств ВНУТРИ кейсов.
Порог отвечает на вопрос «похожи ли эти трое сильнее, чем типичная
тройка участников на типичном кейсе», а не «говорят ли они о медицине».
опорный порог τ =

In [6]:
def clusters_from_sim(S, tau):
    n = S.shape[0]
    parent = list(range(n))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    for i in range(n):
        for j in range(i + 1, n):
            if S[i, j] >= tau:
                ri, rj = (find(i), find(j))
                if ri != rj:
                    parent[ri] = rj
    lab = [find(i) for i in range(n)]
    _, inv = np.unique(lab, return_inverse=True)
    return inv

def entropy_from_labels(lab):
    _, cnt = np.unique(lab, return_counts=True)
    p = cnt / cnt.sum()
    H = float(-(p * np.log(p)).sum())
    k = len(lab)
    return (H, H / np.log(k) if k > 1 else 0.0, float(np.exp(H)), len(cnt))
rows = []
for (arm, case), idx in groups.items():
    if len(idx) < 2:
        continue
    S = sims[arm, case]
    lab = clusters_from_sim(S, TAU)
    H, Hn, eff, ncl = entropy_from_labels(lab)
    iu = np.triu_indices_from(S, 1)
    rows.append({'arm': arm, 'case': case, 'k': len(idx), 'ср. сходство': float(S[iu].mean()), 'мин. сходство': float(S[iu].min()), 'H': H, 'H норм.': Hn, 'эфф. число содержаний': eff, 'кластеров': ncl})
SE = pd.DataFrame(rows)
SE['кластеров при τ по фону'] = [len(np.unique(clusters_from_sim(sims[r.arm, r.case], TAU_BG))) for r in SE.itertuples()]
show(SE.groupby('arm').agg(кейсов=('case', 'size'), ср_сходство=('ср. сходство', 'mean'), H_норм=('H норм.', 'mean'), эфф_содержаний=('эфф. число содержаний', 'mean'), доля_единого_кластера=('кластеров', lambda s: float((s == 1).mean())), единый_кластер_при_фоне=('кластеров при τ по фону', lambda s: float((s == 1).mean()))), 'Семантическая энтропия по плечам (τ = %.3f)' % TAU)
print()
print('Последняя колонка подтверждает предупреждение из шага 2: при пороге,')
print('откалиброванном по фоновому сходству, ВСЕ кейсы схлопываются в один')
print('кластер и метрика становится константой. Именно поэтому порог считается')
print('по внутрикейсовому распределению.')
print()
print('ПЕРВИЧНАЯ МЕТРИКА этого ноутбука — непрерывное «ср. сходство»: она не')
print('требует порога вообще и потому свободна от этого произвола. Энтропия и')
print('эффективное число содержаний приводятся как производные, сопоставимые')
print('по единицам с k_eff из B9.')



Семантическая энтропия по плечам (τ = 0.505)
------------------------------------------------------------
                  кейсов  ср_сходство  H_норм  эфф_содержаний  доля_единого_кластера  единый_кластер_при_фоне
arm                                                                                                          
P-base               229       0.4159  0.8661          2.7051                 0.0917                   1.0000
P-mixed              229       0.4486  0.7038          2.3607                 0.2227                   1.0000
P-strong             229       0.4258  0.8063          2.5587                 0.1092                   1.0000
P-weak               229       0.4361  0.7594          2.4684                 0.1616                   1.0000
homog deepseekv4     229       0.5664  0.1193          1.2073                 0.8384                   1.0000
homog gemini         229       0.5769  0.1168          1.2078                 0.8428                   1.0000
homog gpt    

In [7]:
KEFF_B9 = {'P-base': 1.387, 'P-weak': 1.345, 'P-mixed': 1.363, 'P-strong': 1.371}
agg = SE.groupby('arm').agg(эфф_содержаний=('эфф. число содержаний', 'mean'), H_норм=('H норм.', 'mean')).reset_index()
agg['гетерогенная'] = agg['arm'].map(lambda a: bool(HETERO.get(a, 0)))
agg['k_eff по меткам (B9)'] = agg['arm'].map(KEFF_B9)
agg['номинально k'] = 3
agg = agg.set_index('arm')
show(agg[['гетерогенная', 'номинально k', 'k_eff по меткам (B9)', 'эфф_содержаний', 'H_норм']], 'Сколько независимых мнений: по меткам против по содержанию')
print()
het = agg[agg['гетерогенная']]['эфф_содержаний']
hom = agg[~agg['гетерогенная']]['эфф_содержаний']
print('H6.3 (проверка манипуляции): гетерогенные %.3f против гомогенных %.3f' % (het.mean(), hom.mean()))
u = stats.mannwhitneyu(het, hom) if len(het) and len(hom) else None
if u is not None:
    print('   Манна-Уитни на уровне плеч: p = %s (n = %d против %d — мощности нет,' % (fmt_p(u.pvalue), len(het), len(hom)))
    print('   смотреть надо на направление и величину)')
print()
print('Проверка на уровне КЕЙСОВ, где мощность есть:')
sub = SE.copy()
sub['гетерогенная'] = sub['arm'].map(lambda a: bool(HETERO.get(a, 0)))
a_, b_ = (sub[sub['гетерогенная']]['эфф. число содержаний'], sub[~sub['гетерогенная']]['эфф. число содержаний'])
print('   гетерогенные: n=%d, среднее %.3f   гомогенные: n=%d, среднее %.3f' % (len(a_), a_.mean(), len(b_), b_.mean()))
print('   Манна-Уитни: p = %s' % fmt_p(stats.mannwhitneyu(a_, b_).pvalue))



Сколько независимых мнений: по меткам против по содержанию
------------------------------------------------------------
                  гетерогенная  номинально k  k_eff по меткам (B9)  эфф_содержаний  H_норм
arm                                                                                       
P-base                    True             3                1.3870          2.7051  0.8661
P-mixed                   True             3                1.3630          2.3607  0.7038
P-strong                  True             3                1.3710          2.5587  0.8063
P-weak                    True             3                1.3450          2.4684  0.7594
homog deepseekv4         False             3                   NaN          1.2073  0.1193
homog gemini             False             3                   NaN          1.2078  0.1168
homog gpt                False             3                   NaN          1.2689  0.1712
homog qwen               False             3                

In [8]:
QROWS = []
for arm, rel in PANEL_REL.items():
    d = rd(rel)
    if d is None:
        continue
    for p in participants(d):
        c = 'r1_raw__' + p
        if c not in d.columns:
            continue
        for v, txt in d[c].dropna().items():
            r = parse_r1(txt)
            if r:
                for q in r['questions']:
                    if q.strip():
                        QROWS.append({'arm': arm, 'case': v, 'pid': p, 'q': q})
QQ = pd.DataFrame(QROWS)
print('вопросов извлечено:', len(QQ))
qvec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words='english')
QX = qvec.fit_transform(QQ['q'])
QX = QX.multiply(1.0 / np.sqrt(QX.multiply(QX).sum(axis=1) + 1e-12)).tocsr()
QTAU = 0.5
rows = []
for (arm, case), idx in QQ.groupby(['arm', 'case'], observed=True).indices.items():
    sub = QQ.iloc[idx]
    pids = sub['pid'].unique()
    if len(pids) < 2:
        continue
    M = QX[idx]
    Sm = (M @ M.T).toarray()
    pid_arr = sub['pid'].to_numpy()
    ov = []
    for a, b in combinations(pids, 2):
        ia = np.where(pid_arr == a)[0]
        ib = np.where(pid_arr == b)[0]
        if not len(ia) or not len(ib):
            continue
        block = Sm[np.ix_(ia, ib)]
        ov.append(float(((block.max(axis=1) >= QTAU).mean() + (block.max(axis=0) >= QTAU).mean()) / 2))
    if ov:
        rows.append({'arm': arm, 'case': case, 'перекрытие вопросов': float(np.mean(ov)), 'вопросов всего': len(sub)})
QO = pd.DataFrame(rows)
show(QO.groupby('arm').agg(кейсов=('case', 'size'), перекрытие=('перекрытие вопросов', 'mean'), вопросов=('вопросов всего', 'mean')), 'Перекрытие наборов подвопросов между участниками (порог %.2f)' % QTAU)
print()
print('Читается так: доля вопросов участника, для которых у коллеги нашёлся')
print('похожий вопрос. Значение 1.0 означало бы, что все трое спрашивают об')
print('одном и том же; значение 0.3 — что две трети повестки у каждого своя.')


вопросов извлечено: 43447



Перекрытие наборов подвопросов между участниками (порог 0.50)
-------------------------------------------------------------
                  кейсов  перекрытие  вопросов
arm                                           
P-base               229      0.0989   23.9913
P-mixed              229      0.0728   23.9913
P-strong             229      0.0850   23.9913
P-weak               229      0.0573   23.8865
homog deepseekv4     229      0.4048   23.8952
homog gemini         229      0.1260   24.0000
homog gpt            215      0.3312   23.1070
homog qwen           229      0.5332   23.9956

Читается так: доля вопросов участника, для которых у коллеги нашёлся
похожий вопрос. Значение 1.0 означало бы, что все трое спрашивают об
одном и том же; значение 0.3 — что две трети повестки у каждого своя.


In [9]:
def n_extreme(txt):
    try:
        o = json.loads(txt)
    except Exception:
        return np.nan
    uq = o.get('unified_questions') or []
    n = 0
    for q in uq:
        ev = q.get('extreme_views')
        if isinstance(ev, list):
            n += len([e for e in ev if str(e).strip()])
        elif isinstance(ev, str) and ev.strip() and (ev.strip().lower() not in ('none', 'n/a')):
            n += 1
    return float(n)
rows = []
for arm, rel in PANEL_REL.items():
    d = rd(rel)
    if d is None or 'organizer_after_r1_raw' not in d.columns:
        continue
    ne = d['organizer_after_r1_raw'].map(n_extreme)
    tmp = SE[SE['arm'] == arm].set_index('case')
    ix = tmp.index.intersection(ne.dropna().index)
    if len(ix) < 30:
        continue
    r1, p1 = stats.spearmanr(tmp.loc[ix, 'эфф. число содержаний'], ne.loc[ix])
    r2, p2 = stats.spearmanr(tmp.loc[ix, 'ср. сходство'], ne.loc[ix])
    lab_un = None
    em = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in participants(d)})
    un = (em.nunique(axis=1) == 1).astype(float)
    r3, p3 = stats.spearmanr(un.loc[ix], ne.loc[ix])
    rows.append({'плечо': arm, 'кейсов': len(ix), 'ср. крайних мнений': float(ne.loc[ix].mean()), 'rho (эфф. содержаний)': r1, 'p': p1, 'rho (сходство)': r2, 'p ': p2, 'rho (единогласие метки)': r3, 'p  ': p3})
V = pd.DataFrame(rows).set_index('плечо')
show(V, 'Согласуются ли наши метрики с тем, что организатор пометил крайним')
print()
print('Что тут важно. Если «эфф. содержаний» коррелирует с числом крайних')
print('мнений СИЛЬНЕЕ, чем единогласие метки, значит содержательная метрика')
print('улавливает расхождения, невидимые на уровне меток. Это и есть')
print('центральное утверждение F6, проверенное независимым индикатором.')



Согласуются ли наши метрики с тем, что организатор пометил крайним
------------------------------------------------------------------
                  кейсов  ср. крайних мнений  rho (эфф. содержаний)      p  rho (сходство)     p   rho (единогласие метки)    p  
плечо                                                                                                                            
P-base               227              1.9471                -0.0589 0.3775         -0.0047 0.9436                   0.0028 0.9671
P-weak               229              0.7511                 0.1730 0.0087         -0.1704 0.0098                  -0.3814 0.0000
P-mixed              150              0.6600                 0.2179 0.0074         -0.1754 0.0318                  -0.1819 0.0259
P-strong             136              1.1765                 0.0380 0.6602         -0.0971 0.2608                  -0.3740 0.0000
homog gpt            201              1.2388                 0.1486 0.0353         -0

In [10]:
rows = []
for arm in PANEL_REL:
    if arm not in A or arm not in COMPOSITION:
        continue
    members = ['baseline ' + m for m in COMPOSITION[arm]]
    if not all((m in A for m in members)):
        continue
    mat = pd.concat([js(A, m).rename(m) for m in members], axis=1).dropna()
    out = js(A, arm)
    tmp = SE[SE['arm'] == arm].set_index('case')
    d = rd(PANEL_REL[arm])
    em = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in participants(d)})
    un = (em.nunique(axis=1) == 1).astype(float)
    ix = tmp.index.intersection(mat.index).intersection(out.index).intersection(un.index)
    if len(ix) < 40:
        continue
    gain = out.loc[ix] - mat.loc[ix].mean(axis=1)
    gain_max = out.loc[ix] - mat.loc[ix].max(axis=1)
    qo = QO[QO['arm'] == arm].set_index('case')['перекрытие вопросов'].reindex(ix)
    df = pd.DataFrame({'gain': gain, 'gain_max': gain_max, 'эфф_содержаний': tmp.loc[ix, 'эфф. число содержаний'], 'сходство': tmp.loc[ix, 'ср. сходство'], 'единогласие_метки': un.loc[ix], 'перекрытие_вопросов': qo}).dropna()
    for pred in ['эфф_содержаний', 'сходство', 'единогласие_метки', 'перекрытие_вопросов']:
        r, p = stats.spearmanr(df[pred], df['gain'])
        rows.append({'плечо': arm, 'предиктор': pred, 'n': len(df), 'rho с приростом к mean': r, 'p': p})
P = pd.DataFrame(rows)
show(P.pivot_table(index='предиктор', columns='плечо', values='rho с приростом к mean'), 'Ранговая корреляция метрик разнообразия с приростом панели')
show(P.groupby('предиктор').agg(средняя_rho=('rho с приростом к mean', 'mean'), значимых=('p', lambda s: int((s < 0.05).sum())), плеч=('p', 'size')), 'Сводка по предикторам')



Ранговая корреляция метрик разнообразия с приростом панели
------------------------------------------------------------
плечо                P-base  P-mixed  P-strong  P-weak  homog deepseekv4  homog gemini  homog gpt  homog qwen
предиктор                                                                                                    
единогласие_метки   -0.0214  -0.0083   -0.0785 -0.0782           -0.0800       -0.0124     0.0125     -0.0448
перекрытие_вопросов -0.0184   0.2128    0.0430  0.0963           -0.0946        0.0662     0.1099      0.1416
сходство            -0.0415  -0.1556   -0.1442 -0.1722           -0.1050       -0.1711     0.0417     -0.0101
эфф_содержаний       0.0683   0.1146    0.0682  0.1334            0.1122        0.1762    -0.0810     -0.0225

Сводка по предикторам
------------------------------------------------------------
                     средняя_rho  значимых  плеч
предиктор                                       
единогласие_метки        -0.0389     

In [11]:
parts = []
for arm in PANEL_REL:
    if arm not in A or arm not in COMPOSITION:
        continue
    members = ['baseline ' + m for m in COMPOSITION[arm]]
    if not all((m in A for m in members)):
        continue
    mat = pd.concat([js(A, m).rename(m) for m in members], axis=1).dropna()
    out = js(A, arm)
    tmp = SE[SE['arm'] == arm].set_index('case')
    d = rd(PANEL_REL[arm])
    em = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in participants(d)})
    un = (em.nunique(axis=1) == 1).astype(float)
    ix = tmp.index.intersection(mat.index).intersection(out.index).intersection(un.index)
    if len(ix) < 40:
        continue
    parts.append(pd.DataFrame({'gain': (out.loc[ix] - mat.loc[ix].mean(axis=1)).to_numpy(), 'eff': tmp.loc[ix, 'эфф. число содержаний'].to_numpy(), 'unan': un.loc[ix].to_numpy(), 'arm': arm, 'case': ix}))
LL = pd.concat(parts, ignore_index=True)
n0 = len(LL)
LL = LL.replace([np.inf, -np.inf], np.nan).dropna(subset=['gain', 'eff', 'unan'])
print('строк: %d (отброшено с пропусками: %d)' % (len(LL), n0 - len(LL)))
LL = LL.reset_index(drop=True)
LL['eff_z'] = (LL['eff'] - LL['eff'].mean()) / LL['eff'].std(ddof=0)
D = pd.get_dummies(LL['arm'], prefix='a', drop_first=True).astype(float)
BASE = pd.concat([LL[['gain', 'eff_z', 'unan']].astype(float), D], axis=1)
ok = np.isfinite(BASE.to_numpy()).all(axis=1)
print('строк в модели: %d из %d' % (int(ok.sum()), len(BASE)))
BASE, gg = (BASE[ok], LL.loc[ok, 'case'].astype('category').cat.codes)
for spec, lab in [(['eff_z'], 'только содержательная'), (['unan'], 'только меточная'), (['eff_z', 'unan'], 'обе')]:
    X_ = sm.add_constant(BASE[spec + list(D.columns)])
    m = sm.OLS(BASE['gain'], X_).fit(cov_type='cluster', cov_kwds={'groups': gg})
    print('%-24s ' % lab, end='')
    for k in spec:
        print('%s: коэф %+.4f p %s   ' % (k, m.params[k], fmt_p(m.pvalues[k])), end='')
    print('  R² = %.4f  n = %d' % (m.rsquared, int(m.nobs)))


строк: 1814 (отброшено с пропусками: 0)
строк в модели: 1814 из 1814
только содержательная    eff_z: коэф +0.0177 p 0.007     R² = 0.1273  n = 1814
только меточная          unan: коэф -0.0176 p 0.171     R² = 0.1227  n = 1814
обе                      eff_z: коэф +0.0168 p 0.010   unan: коэф -0.0132 p 0.298     R² = 0.1281  n = 1814


In [12]:
HET = {a: bool(HETERO.get(a, 0)) for a in PANEL_REL}
rows = []
for tau in [0.2, 0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7]:
    eff_h, eff_o = ([], [])
    for key, S in sims.items():
        lab = clusters_from_sim(S, tau)
        _, _, eff, _ = entropy_from_labels(lab)
        (eff_h if HET.get(key[0]) else eff_o).append(eff)
    rows.append({'τ': tau, 'гетерогенные': float(np.mean(eff_h)) if eff_h else np.nan, 'гомогенные': float(np.mean(eff_o)) if eff_o else np.nan, 'разрыв': float(np.mean(eff_h)) - float(np.mean(eff_o)) if eff_h and eff_o else np.nan})
T = pd.DataFrame(rows).set_index('τ')
T['гетеро / гомо'] = T['гетерогенные'] / T['гомогенные']
show(T, 'Чувствительность к порогу: эффективное число содержаний')
print()
print('опорный τ = %.3f (медиана внутрикейсовых сходств)' % TAU)
print()
print('ЧЕСТНЫЙ ВЫВОД ПО ЧУВСТВИТЕЛЬНОСТИ.')
print('АБСОЛЮТНОЕ значение эффективного числа содержаний СИЛЬНО зависит от τ:')
print('при 0.30 это почти 1.0, при 0.55 уже заметно больше 2. Поэтому цифру')
print('«2.7 против 1.39» нельзя подавать как точечную оценку.')
print()
print('Что УСТОЙЧИВО и потому может быть заявлено:')
print('  1) непрерывное среднее сходство — оно вообще не требует порога:')
print('     гетерогенные %.3f против гомогенных %.3f;' % (SE[SE['arm'].map(HET)]['ср. сходство'].mean(), SE[~SE['arm'].map(HET)]['ср. сходство'].mean()))
print('  2) отношение «гетеро / гомо» держится выше 1 на всём диапазоне τ,')
print('     где кластеризация вообще что-то различает;')
print('  3) перекрытие наборов вопросов (шаг 5) — там порог влияет мало,')
print('     потому что разрыв между плечами на порядок больше.')



Чувствительность к порогу: эффективное число содержаний
------------------------------------------------------------
        гетерогенные  гомогенные  разрыв  гетеро / гомо
τ                                                      
0.2000        1.0019      1.0000  0.0019         1.0019
0.3000        1.0308      1.0000  0.0308         1.0308
0.4000        1.4643      1.0059  0.4584         1.4557
0.4500        1.9544      1.0396  0.9148         1.8800
0.5000        2.4774      1.1660  1.3114         2.1246
0.5500        2.8304      1.5289  1.3015         1.8513
0.6000        2.9621      1.9704  0.9916         1.5033
0.7000        2.9989      2.8260  0.1729         1.0612

опорный τ = 0.505 (медиана внутрикейсовых сходств)

ЧЕСТНЫЙ ВЫВОД ПО ЧУВСТВИТЕЛЬНОСТИ.
АБСОЛЮТНОЕ значение эффективного числа содержаний СИЛЬНО зависит от τ:
при 0.30 это почти 1.0, при 0.55 уже заметно больше 2. Поэтому цифру
«2.7 против 1.39» нельзя подавать как точечную оценку.

Что УСТОЙЧИВО и потому может быть заяв

In [13]:
def jaccard_words(a, b):
    sa = set(re.findall('[a-z]{4,}', a.lower()))
    sb = set(re.findall('[a-z]{4,}', b.lower()))
    return len(sa & sb) / max(len(sa | sb), 1)
rng = np.random.default_rng(1)
samp = list(sims)[:400]
rows = []
for key in samp:
    idx = groups[key]
    txts = R1.iloc[idx]['full_text'].tolist()
    if len(txts) < 2:
        continue
    js_ = [jaccard_words(a, b) for a, b in combinations(txts, 2)]
    ws = [abs(words(a) - words(b)) for a, b in combinations(txts, 2)]
    S = sims[key]
    rows.append({'косинус TF-IDF': float(S[np.triu_indices_from(S, 1)].mean()), 'Жаккар по словам': float(np.mean(js_)), 'разница длин, слов': float(np.mean(ws))})
AL = pd.DataFrame(rows)
show(AL.describe().T[['mean', 'std', '25%', '50%', '75%']], 'Три меры сходства ответов (выборка %d кейсов)' % len(AL))
r_j = stats.spearmanr(AL['косинус TF-IDF'], AL['Жаккар по словам']).statistic
r_l = stats.spearmanr(AL['косинус TF-IDF'], AL['разница длин, слов']).statistic
print()
print('корреляция косинуса и Жаккара:      rho = %.3f' % r_j)
print('корреляция косинуса и разницы длин: rho = %.3f' % r_l)
print()
print('ЧТО ЭТО ЗНАЧИТ — БЕЗ ПРИУКРАШИВАНИЯ.')
print('Корреляция с разницей длин низкая (%.3f) — это хорошо: мера НЕ сводится' % r_l)
print('к «кто написал длиннее», то есть не повторяет уже известный из C13 и B24')
print('эффект длины.')
print()
print('Но корреляция с Жаккаром всего %.3f — это НЕ высокая величина, и её' % r_j)
print('надо признать честно. Две лексические меры одного и того же согласуются')
print('слабо, значит выбор способа измерения лексического пересечения влияет на')
print('результат сильнее, чем хотелось бы. Причина понятна: TF-IDF взвешивает')
print('редкие термины, Жаккар считает все слова одинаково, и на текстах, где')
print('90%% лексики общая (одна виньетка), эти два взгляда расходятся.')
print()
print('Практический вывод: покейсные значения метрики шумны, и опираться на них')
print('по отдельности нельзя. Устойчивы только АГРЕГАТЫ по плечам (сотни')
print('кейсов) и сравнения между плечами — то есть ровно то, на чём построены')
print('выводы шагов 3-5. Точечные утверждения вида «на этом кейсе участники')
print('разошлись сильно» этот инструмент не поддерживает.')



Три меры сходства ответов (выборка 400 кейсов)
------------------------------------------------------------
                      mean     std     25%     50%      75%
косинус TF-IDF      0.4309  0.0656  0.3872  0.4290   0.4788
Жаккар по словам    0.2958  0.0299  0.2765  0.2958   0.3149
разница длин, слов 81.2567 49.2000 46.0000 74.0000 104.1667

корреляция косинуса и Жаккара:      rho = 0.253
корреляция косинуса и разницы длин: rho = 0.106

ЧТО ЭТО ЗНАЧИТ — БЕЗ ПРИУКРАШИВАНИЯ.
Корреляция с разницей длин низкая (0.106) — это хорошо: мера НЕ сводится
к «кто написал длиннее», то есть не повторяет уже известный из C13 и B24
эффект длины.

Но корреляция с Жаккаром всего 0.253 — это НЕ высокая величина, и её
надо признать честно. Две лексические меры одного и того же согласуются
слабо, значит выбор способа измерения лексического пересечения влияет на
результат сильнее, чем хотелось бы. Причина понятна: TF-IDF взвешивает
редкие термины, Жаккар считает все слова одинаково, и на текстах, где


In [14]:
save(SE, 'semantic_response_diversity_semantic_entropy.csv')
save(QO, 'semantic_response_diversity_question_overlap.csv')
save(agg.reset_index(), 'semantic_response_diversity_keff_comparison.csv')
save(V.reset_index(), 'semantic_response_diversity_organizer_validation.csv')
save(T.reset_index(), 'semantic_response_diversity_tau_sensitivity.csv')


сохранено: out/semantic_response_diversity_semantic_entropy.csv  (1818 строк)
сохранено: out/semantic_response_diversity_question_overlap.csv  (1818 строк)
сохранено: out/semantic_response_diversity_keff_comparison.csv  (8 строк)
сохранено: out/semantic_response_diversity_organizer_validation.csv  (8 строк)
сохранено: out/semantic_response_diversity_tau_sensitivity.csv  (8 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\panel_mechanisms\\results\\semantic_response_diversity_tau_sensitivity.csv'